# Sistema di raccomandazione

In [ ]:
# Preconfigurazione
from pymongo import MongoClient
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import string
import random

client = MongoClient("mongodb://localhost:27017/")
db = client["recommender_system"]
ratings_collection = db["ratings"]

In [ ]:
# Funzione per generare ID casuali
def generate_random_id(prefix="", length=6):
    return prefix +''.join(random.choices(string.ascii_lowercase + string.digits, k=length))

In [ ]:
#Funzione per generare documenti di validazione
def generate_random_ratings(num_ratings=1000, num_users=100, num_items=50):
    # Creiamo prima l'insieme fisso di utenti e items
    users = [generate_random_id("user_") for _ in range (num_users)]
    items = [generate_random_id("item_") for _ in range (num_items)]

    ratings = []
    used_pairs = set() # per tracciare le combinazioni già usate

    while len(ratings) < num_ratings:
        user_id = random.choice(users)
        item_id = random.choice(items)

        if(user_id, item_id) not in used_pairs:
            rating = random.randint(1,5)
            ratings.append({
                "user_id": user_id,
                "item_id": item_id,
                "rating": rating
            })
            used_pairs.add((user_id, item_id)) # aggiunto la combinazione per non ripeterla

        if len(used_pairs) > num_items * num_users:
            break
    
    return ratings

In [ ]:
# Funzione per inserire i documenti a DB
def insert_random_ratings(collection, num_ratings=1000, num_users=100, num_items=50):
    ratings = generate_random_ratings(num_ratings, num_users, num_items)
    collection.insert_many(ratings)
    print(f"Inseriti {len(ratings)} valori in {collection}")

In [ ]:
#Funzione per ottenere la matrice delle valutazioni (user-item matrix)
def get_ratings_matrix(collection):
    # Estraggo tutti i ratings
    ratings = list(collection.find())

    # Estraggo utenti e oggetti unici
    #users = list(set([r['user_id'] for r in ratings]))
    users = collection.distinct("user_id")

    #items = list(set([r['item_id'] for r in ratings]))
    items = collection.distinct("item_id")

    # Creo matrice utenti x oggetti
    ratings_matrix = np.zeros((len(users), len(items)))

    # Creazione dei dizionari per mappare utenti e oggetti agli indici - associo utente e oggetto a indice numerico
    user_index = {user: idx for idx, user in enumerate(users)}
    item_index = {item: idx for idx, item in enumerate(items)}

    for rating in ratings:
        user_idx = user_index[rating['user_id']]
        item_idx = item_index[rating['item_id']]
        ratings_matrix[user_idx, item_idx] = rating['rating']
    
    return ratings_matrix, user_index, item_index

In [ ]:
# Funzione per calcolare la similarità degli oggetti
def calculate_item_similarity(ratings_matrix):
    # Valuto trasposta della matrice per avere gli item come righe
    item_ratings = ratings_matrix.T

    # Calcolo somiglianza coseno tra oggetti
    similarity_matrix = cosine_similarity(item_ratings)

    return similarity_matrix

In [ ]:
#Funzione per calcolare similarità tra utenti
def calculate_user_similarity(ratings_matrix):
    # Le righe sono già gli utenti
    similarity_matrix = cosine_similarity(ratings_matrix)

    return similarity_matrix

In [ ]:
# Raccomandazione USER-BASED

# Funzione per predire la valutazione di un utente per un oggetto (partendo dagli altri utenti che hanno valutato gli stessi oggetti)
def predict_ratings_user_based(user_id, item_id, ratings_matrix, user_similarity_matrix, user_index, item_index):
    user_idx = user_index[user_id]
    item_idx = item_index[item_id]

    # Similarità tra l'utente target e tutti gli altri utenti
    user_similarities = user_similarity_matrix[user_idx]

    # Ratings degli altri utenti per l'item target
    item_ratings = ratings_matrix[:,item_idx]

    # Considero solo gli utenti che hanno valutato l'item
    mask = item_ratings > 0
    if np.sum(mask) == 0:
        return 0 # Nessun rating disponibile
    
    numerator = np.dot(user_similarities[mask], item_ratings[mask])
    denominator = np.sum(np.abs(user_similarities[mask]))

    if denominator == 0:
        return 0
    
    predicted_rating = numerator / denominator
    return predicted_rating
    

In [37]:
# Raccomandazione ITEM-BASED

# Funzione per predire la valutazione di un utente per un oggetto (partendo dagli altri utenti che hanno valutato gli stessi oggetti)
def predict_ratings_item_based(user_id, item_id, ratings_matrix, item_similarity_matrix, user_index, item_index):
    user_idx = user_index[user_id]
    item_idx = item_index[item_id]

    # Similarità tra l'utente target e tutti gli altri utenti
    item_similarities = item_similarity_matrix[item_idx]

    # Ratings degli altri utenti per l'item target
    user_ratings = ratings_matrix[user_idx]

    # Considero solo gli utenti che hanno valutato l'item
    mask = user_ratings > 0
    if np.sum(mask) == 0:
        return 0 # Nessun rating disponibile
    
    numerator = np.dot(item_similarities[mask], user_ratings[mask])
    denominator = np.sum(np.abs(item_similarities[mask]))

    if denominator == 0:
        return 0
    
    predicted_rating = numerator / denominator
    return predicted_rating

# MAIN - USER BASE

In [ ]:
# Pulizia pre-test
ratings_collection.drop()

In [ ]:
# Main program - USER BASED

# 0. Inserisco valori
insert_random_ratings(ratings_collection, 1000, 100, 50)

# 1. Ottieni la matrice delle valutazioni
ratings_matrix, user_index, item_index = get_ratings_matrix(ratings_collection)

# 2. Calcola la matrice di similarità tra gli oggetti
user_similarity_matrix = calculate_user_similarity(ratings_matrix)

Inseriti 1000 valori in Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'recommender_system'), 'ratings')


In [32]:
# 3. Predici una valutazione - USED BASED
user_id = "user_1yr1k9"
item_id = "item_mmfz7l"
# rating = 2
# predetto = 2.77
predicted_rating = predict_ratings_user_based(user_id, item_id, ratings_matrix, user_similarity_matrix, user_index, item_index)
print(f"Valutazione predetta per l'utente {user_id} sull'oggetto {item_id} = {predicted_rating:.2f}")

Valutazione predetta per l'utente user_1yr1k9 sull'oggetto item_mmfz7l = 3.06


# MAIN - ITEM BASED

In [41]:
# Pulizia pre-test
ratings_collection.drop()

In [ ]:
# Main program - ITEM BASED

# 0. Inserisco valori
insert_random_ratings(ratings_collection, 1000, 100, 50)

# 1. Ottieni la matrice delle valutazioni
ratings_matrix, user_index, item_index = get_ratings_matrix(ratings_collection)

# 2. Calcola la matrice di similarità tra gli oggetti
item_similarity_matrix = calculate_item_similarity(ratings_matrix)

Inseriti 1000 valori in Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'recommender_system'), 'ratings')


In [ ]:
# 3. Predici una valutazione - ITEM BASED
user_id = "user_mymz3d"
item_id = "item_z6zpwu"
# rating = 3
# predetto = 2.61
predicted_rating = predict_ratings_item_based(user_id, item_id, ratings_matrix, item_similarity_matrix, user_index, item_index)
print(f"Valutazione predetta per l'utente {user_id} sull'oggetto {item_id} = {predicted_rating:.2f}")

Valutazione predetta per l'utente user_mymz3d sull'oggetto item_z6zpwu = 2.61


# Sistema User-Based + Item-based